# 실습 3: Amazon AgentCore Identity로 Agent 만들기 

## 개요

이 실습에서는 Amazon Bedrock AgentCore Identity 기능을 통합하여 기존 고객 지원 Agent를 강화합니다. 이를 통해 AgentCore의 Identity Provider로 자격 증명을 적절하게 관리하면서 OAuth2 흐름을 사용하여 Google Calendar 같은 외부 서비스에 안전하게 인증할 수 있습니다.

Calendar 통합을 통해 지원 Agent는 고객과의 대화 내에서 제품 시연 및 기술 지원 예약 같은 이벤트를 직접 예약할 수 있으므로 지원 워크플로를 간소화하고 전반적인 고객 경험을 개선할 수 있습니다.

![Agent 아키텍처](images/architecture_lab6_identity.png)

**기반 예제**: [공식 Customer Support Assistant](https://github.com/awslabs/amazon-bedrock-agentcore-samples/tree/main/02-use-cases/customer-support-assistant)


### 실습 세부 정보

| 정보        | 세부 정보                                                                        |
| :----------------- |:-------------------------------------------------------------------------------|
| 실습 유형           | 점진적 기능 향상                                                        |
| Agent 유형         | 단일                                                                         |
| Agentic 프레임워크  | Strands Agents                                                                 |
| LLM 모델          | Amazon Nova 2 Lite                                                             |
| 실습 구성 요소     | AgentCore Identity, OAuth2 Provider, Google Calendar API, Cognito 통합 |
| 실습 분야       | 고객 지원                                                               |
| 예제 난이도 | 보통                                                                       |
| 사용 SDK           | Amazon BedrockAgentCore Python SDK, Strands Agents, Google API Client          |

### 실습 아키텍처

이 실습에서는 고객 지원 Agent에 Identity 관리 기능을 추가합니다. Agent는 AgentCore Identity Provider를 통해 사용자를 인증하고, 인증된 사용자를 대신하여 Google Calendar 같은 외부 서비스에 액세스합니다.

Agent는 다음 항목을 통합합니다.
- **AgentCore Identity**: 안전한 자격 증명 관리 및 OAuth2 흐름
- **Google OAuth2 Provider**: Google 서비스 인증
- **Calendar 도구**: 이벤트 생성 및 Calendar 정보 검색
- **Cognito Provider**(선택 사항): 사용자 지정 Identity Provider 통합


### 실습 주요 기능

- 안전한 OAuth2 인증 흐름
- 적절한 자격 증명 관리를 통한 외부 서비스 통합
- Google Calendar API 통합
- 사용자 지정 Identity Provider 구성

## 사전 요구 사항

이 튜토리얼을 실행하려면 다음 항목이 필요합니다.

- Python 3.10+
- 구성된 AWS 자격 증명
- Amazon Bedrock AgentCore SDK
- Strands Agents
- **실습 1 및 2 완료** - 이 실습은 기존 Agent를 기반으로 진행됩니다.
- **Google Developer Console 액세스** - OAuth2 자격 증명 생성에 필요
- **AgentCore Identity 권한** - AgentCore Identity 액세스 권한이 있는 IAM Role

**참고**: 계속하기 전에 실습 1과 2에서 만든 기본 Agent가 올바르게 작동하는지 확인하세요.


## 단계 1: 종속성 설치 및 라이브러리 가져오기

AgentCore Identity 통합, Google API 액세스, OAuth2 인증 흐름에 필요한 패키지를 설치합니다. 이후에 사용할 도우미 함수도 생성합니다.

In [ ]:
# 필수 패키지 설치
%pip install strands-agents strands-agents-tools "boto3>=1.39.15" python-dotenv utils google-auth google-api-python-client ddgs -q

In [ ]:
# 라이브러리 가져오기
import boto3
import click
import sys
import json
import os
from botocore.exceptions import ClientError

from bedrock_agentcore.identity.auth import requires_access_token
from google.oauth2.credentials import Credentials
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from datetime import datetime, timedelta
from strands import tool
from strands import Agent
from strands.models import BedrockModel
import webbrowser

from lab_helpers.utils import get_ssm_parameter

session = boto3.session.Session()
region = session.region_name

identity_client = boto3.client(
    "bedrock-agentcore-control",
    region_name=region,
)
cognito = boto3.client("cognito-idp")
ssm = boto3.client("ssm", region_name=region)

print("✅ Libraries imported successfully!")

In [ ]:
# SSM에서 Provider 이름을 저장, 검색, 삭제하는 도우미 함수


def store_provider_name_in_ssm(provider_name: str):
    """자격 증명 공급자 이름을 SSM 파라미터에 저장합니다."""
    param_name = "/app/customersupport/agentcore/google_provider"
    try:
        ssm.put_parameter(Name=param_name, Value=provider_name, Type="String", Overwrite=True)
        click.echo(f"🔐 Stored provider name in SSM: {param_name}")
    except ClientError as e:
        click.echo(f"⚠️ Failed to store provider name in SSM: {e}")


def get_provider_name_from_ssm() -> str:
    """SSM 파라미터에서 자격 증명 공급자 이름을 가져옵니다."""
    param_name = "/app/customersupport/agentcore/google_provider"
    try:
        response = ssm.get_parameter(Name=param_name)
        return response["Parameter"]["Value"]
    except ClientError:
        return None


def delete_ssm_param():
    """공급자용 SSM 파라미터를 삭제합니다."""
    param_name = "/app/customersupport/agentcore/google_provider"
    try:
        ssm.delete_parameter(Name=param_name)
        click.echo(f"🧹 Deleted SSM parameter: {param_name}")
    except ClientError as e:
        click.echo(f"⚠️ Failed to delete SSM parameter: {e}")

## 단계 2: 고객 지원 도구 가져오기

Agent의 핵심 기능을 유지하기 위해 실습 1의 고객 지원 도구를 재사용합니다.


## 단계 3: AgentCore Identity 클라이언트 구성

이 단계에서는 Google Calendar API 기능을 고객 지원 Agent에 통합합니다. 사용자를 대신하여 Google Calendar 같은 외부 서비스에 안전하게 액세스하려면 적절한 인증 메커니즘을 구현해야 합니다.

Amazon Bedrock AgentCore Identity는 OAuth2 인증 흐름을 간소하게 관리하는 방식을 제공하여 수동 토큰 관리, 갱신 처리, 자격 증명 저장의 복잡성을 줄여 줍니다. 이 서비스는 Agent와 외부 서비스 공급자 사이에서 안전한 중개자 역할을 합니다.

Google OAuth 클라이언트를 생성하려면 다음 단계를 따르세요.

#### ✅ 1. Google Developer Console에서 프로젝트 생성

1. [Google Developer Console](https://console.developers.google.com/)로 이동합니다.
2. 상단 탐색 모음에서 “Create Project”를 클릭합니다.
3. Project Name을 입력합니다.
4. Organization을 선택하거나 해당하지 않는 경우 “No organization”으로 둡니다.
5. Create를 클릭합니다.

새 프로젝트가 프로젝트 목록에 표시됩니다.

#### 📦 2. Google Calendar API 활성화

1. 프로젝트를 선택한 상태에서 왼쪽 메뉴를 열고 APIs & Services > Library로 이동합니다.
2. 검색 창에 Google Calendar API를 입력합니다.
3. 검색 결과에서 Google Calendar API를 클릭합니다.
4. Enable을 클릭합니다.

#### 🛡️ 3. OAuth Consent Screen 구성

1. 왼쪽 메뉴에서 APIs & Services > OAuth consent screen으로 이동합니다.

2. “Get started”를 클릭합니다.

3. 필수 필드인 App Name과 User Support Email을 입력합니다.
4. Next를 클릭한 다음 User Type(Internal 또는 External)을 선택합니다. External을 선택한 경우 테스터 이메일 주소를 추가합니다. Developer Contact Information에 이메일 주소를 입력합니다.
5. 약관에 동의하고 Finish를 클릭합니다.
6. Create를 클릭하여 consent screen 구성을 완료합니다.

#### 🔧 4. OAuth 2.0 자격 증명 생성

1. 왼쪽 메뉴에서 APIs & Services > Credentials로 이동합니다.
2. Create Credentials > OAuth client ID를 클릭합니다.
3. 애플리케이션 유형으로 Web application을 선택합니다.
4. 자격 증명 이름을 입력합니다.
5. Authorized redirect URIs 아래에 다음 redirect URI를 추가합니다.
   - `https://bedrock-agentcore.us-east-1.amazonaws.com/identities/oauth2/callback`
6. Create를 클릭합니다.

#### 🔑 5. Client ID 및 Client Secret 가져오기

생성이 완료되면 대화 상자에 Client ID와 Client Secret이 표시됩니다. Download JSON을 클릭하여 자격 증명을 파일로 저장합니다. 이 파일을 프로젝트에 `credentials.json`이라는 이름으로 저장합니다. 이름이 다르면 파일명을 변경해야 할 수 있습니다. 

#### 🔍 6. Data Access Scope 업데이트

1. APIs & Services > Credentials로 이동합니다.
2. 생성한 OAuth 2.0 client ID를 클릭합니다.
3. 왼쪽 메뉴에서 Data access를 선택합니다.
4. “Add or remove scopes”를 클릭합니다.
5. Manually add scopes 아래에 `https://www.googleapis.com/auth/calendar` scope를 입력합니다.
6. Update를 클릭한 다음 Save를 클릭하여 구성을 확인합니다.

In [ ]:
credentials_file = "credentials.json"

# 자격 증명 파일의 형식을 확인하고 자격 증명 추출

if not os.path.isfile(credentials_file):
    print(f"❌ Error: '{credentials_file}' file not found")
    sys.exit(1)

print(f"📄 Reading credentials from {credentials_file}...")
try:
    with open(credentials_file, "r") as f:
        data = json.load(f)
except json.JSONDecodeError as e:
    print(f"❌ Error parsing JSON: {e}")
    sys.exit(1)

web_config = data.get("web")
if not web_config:
    print("❌ Error: 'web' section missing in credentials.json")
    sys.exit(1)

client_id = web_config.get("client_id")
client_secret = web_config.get("client_secret")

if not client_id:
    print("❌ Error: 'client_id' not found in credentials.json")
    sys.exit(1)

if not client_secret:
    print("❌ Error: 'client_secret' not found in credentials.json")
    sys.exit(1)

print("✅ Client ID and Secret loaded from credentials.json")

In [ ]:
google_provider_name = "customersupport-google-calendar"

try:
    print("🔧 Creating Google OAuth2 credential provider...")
    google_provider = identity_client.create_oauth2_credential_provider(
        name=google_provider_name,
        credentialProviderVendor="GoogleOauth2",
        oauth2ProviderConfigInput={
            "googleOauth2ProviderConfig": {
                "clientId": client_id,
                "clientSecret": client_secret,
            }
        },
    )

    print("✅ Google OAuth2 credential provider created successfully")
    google_provider_arn = google_provider["credentialProviderArn"]
    print(f"   Provider ARN: {google_provider_arn}")
    print(f"   Provider Name: {google_provider['name']}")

    # SSM에 Provider 이름 저장
    store_provider_name_in_ssm(google_provider_name)
except Exception as e:
    print(f"❌ Error creating Google credential provider: {str(e)}")

In [ ]:
# 모든 OAuth2 자격 증명 Provider 나열
try:
    response = identity_client.list_oauth2_credential_providers(maxResults=20)
    providers = response.get("credentialProviders", [])
    print(providers)
except Exception as e:
    print(f"❌ Error listing credential providers: {str(e)}", err=True)

이제 AgentCore Identity를 사용하여 Google Calendar에 인증하고 사용자를 대신해 Calendar 작업을 수행하는 도구를 생성합니다.

먼저 OAuth2 인증 흐름을 구성합니다.

In [ ]:
async def on_auth_url(url: str):
    webbrowser.open(url)


SCOPES = ["https://www.googleapis.com/auth/calendar"]

google_access_token = None


@requires_access_token(
    provider_name=google_provider_name,
    scopes=["https://www.googleapis.com/auth/calendar"],  # Google OAuth2 범위
    auth_flow="USER_FEDERATION",  # On-behalf-of user(3LO) 흐름
    on_auth_url=on_auth_url,  # authorization URL을 console에 출력
    force_authentication=True,
    into="access_token",
)
def get_google_access_token(access_token: str):
    return access_token

이제 Calendar 관리 도구를 생성합니다.

In [ ]:
@tool(
    name="Create_calendar_event",
    description="Creates a new event on your Google Calendar",
)
def create_calendar_event() -> str:
    google_access_token = ""
    try:
        google_access_token = get_google_access_token(access_token=google_access_token)
        if not google_access_token:
            raise Exception("requires_access_token did not provide tokens")
    except Exception as e:
        return "Error Authentication with Google: " + str(e)

    creds = Credentials(token=google_access_token, scopes=SCOPES)

    try:
        service = build("calendar", "v3", credentials=creds)

        # 이벤트 세부 정보 정의
        start_time = datetime.now() + timedelta(hours=1)
        end_time = start_time + timedelta(hours=1)

        event = {
            "summary": "Test Event from API",
            "location": "Virtual",
            "description": "This event was created using the Google Calendar API.",
            "start": {
                "dateTime": start_time.isoformat() + "Z",  # UTC 시간
                "timeZone": "UTC",
            },
            "end": {
                "dateTime": end_time.isoformat() + "Z",
                "timeZone": "UTC",
            },
        }

        created_event = service.events().insert(calendarId="primary", body=event).execute()

        return json.dumps(
            {
                "event_created": True,
                "event_id": created_event.get("id"),
                "htmlLink": created_event.get("htmlLink"),
            }
        )

    except HttpError as error:
        return json.dumps({"error": str(error), "event_created": False})
    except Exception as e:
        return json.dumps({"error": str(e), "event_created": False})

In [ ]:
@tool(
    name="Get_calendar_events_today",
    description="Retrieves the calendar events for the day from your Google Calendar",
)
def get_calendar_events_today() -> str:
    google_access_token = ""
    try:
        google_access_token = get_google_access_token(access_token=google_access_token)

        if not google_access_token:
            raise Exception("requires_access_token did not provide tokens")
    except Exception as e:
        return "Error Authentication with Google: " + str(e)

    # 제공된 access token으로 자격 증명 생성
    creds = Credentials(token=google_access_token, scopes=SCOPES)
    try:
        service = build("calendar", "v3", credentials=creds)
        # Calendar API 호출
        today_start = datetime.now().replace(hour=0, minute=0, second=0, microsecond=0)
        today_end = today_start.replace(hour=23, minute=59, second=59)

        # CDT 시간대(-05:00) 형식 적용
        timeMin = today_start.strftime("%Y-%m-%dT00:00:00-05:00")
        timeMax = today_end.strftime("%Y-%m-%dT23:59:59-05:00")

        events_result = (
            service.events()
            .list(
                calendarId="primary",
                timeMin=timeMin,
                timeMax=timeMax,
                singleEvents=True,
                orderBy="startTime",
            )
            .execute()
        )
        events = events_result.get("items", [])
        if not events:
            return json.dumps({"events": []})  # 빈 events 배열을 JSON으로 반환

        return json.dumps({"events": events})  # object로 감싼 events 반환
    except HttpError as error:
        error_message = str(error)
        return json.dumps({"error": error_message, "events": []})
    except Exception as e:
        error_message = str(e)
        return json.dumps({"error": error_message, "events": []})

새로운 Calendar 기능을 포함하는 고객 지원 Agent를 생성합니다.

In [ ]:
model_id = "global.amazon.nova-2-lite-v1:0"
model = BedrockModel(
    model_id=model_id,
)
system_prompt = """
    You are a helpful and professional customer support assistant for an electronics e-commerce company.
Your role is to:
- Provide accurate information using the tools available to you
- Support the customer with technical information and product specifications.
- Be friendly, patient, and understanding with customers
- Always offer additional help after answering questions
- If you can't help with something, direct customers to the appropriate contact

You have access to the following tools:
1. get_return_policy() - For warranty and return policy questions
2. get_product_info() - To get information about a specific product
3. web_search() - To access current technical documentation, or for updated information. 
4. create_calendar_event() - To create a new calendar event
5. get_calendar_events_today() - To find events on the calendar today
Always use the appropriate tool to get accurate, up-to-date information rather than making assumptions about electronic products or specifications.
    """
agent = Agent(
    model=model,
    system_prompt=system_prompt,
    tools=[create_calendar_event, get_calendar_events_today],
    callback_handler=None,
)

이제 Agent를 테스트할 차례입니다! ```Polling for token for authorization url```이라는 메시지 다음에 URL이 표시됩니다. 이 URL을 클릭하여 Google 계정에 로그인하고 Agent에 Google Calendar 액세스 권한을 부여합니다.

In [ ]:
print(str(agent("Can you create a new event on my cal? You can call the create_calendar_event directly.")))

In [ ]:
print(str(agent("Whats my agenda for today?")))

## 축하합니다! 🎉

**Amazon AgentCore Identity 기능으로 Agent 만들기**를 성공적으로 완료했습니다!

### 완료한 작업

✅ **AgentCore Identity 구성**: 안전한 인증을 위한 OAuth2 Credential Provider 설정  
✅ **Google Calendar 통합**: Calendar 이벤트 관리 및 조회 도구 생성  
✅ **고객 지원 강화**: Agent에 예약 및 Calendar 기능 추가  

### 주요 학습 내용
- **Identity 관리**: AgentCore Identity를 사용한 안전한 자격 증명 관리
- **OAuth2 흐름**: 사용자 페더레이션 및 토큰 기반 인증 구현
- **외부 API 통합**: Google Calendar 같은 서드 파티 서비스에 안전하게 연결
- **도구 강화**: 기존 Agent 도구에 Identity 인식 기능 추가

## 다음 단계

Agent를 더 강화할 준비가 되었나요? 다음 실습을 계속 진행하세요.

- **실습 4**: Gateway를 활용하여 도구 및 기타 리소스를 안전하게 연결
- **실습 5**: 프로덕션 모니터링을 위한 Observability 및 Guardrail 구현
- **실습 6**: 확장 가능한 프로덕션 호스팅을 위해 AgentCore Runtime에 배포

## 리소스

- [AgentCore Identity 문서](https://docs.aws.amazon.com/bedrock/latest/userguide/agentcore-identity.html)
- [Google Calendar API 문서](https://developers.google.com/calendar/api)
- [Strands Agents 문서](https://github.com/strands-agents/sdk-python)
- [Amazon Bedrock Models](https://docs.aws.amazon.com/bedrock/latest/userguide/models-supported.html)
- [공식 고객 지원 샘플](https://github.com/awslabs/amazon-bedrock-agentcore-samples/tree/main/02-use-cases/customer-support-assistant)

---

**훌륭합니다! 이제 고객 지원 Agent에 안전한 Identity 관리 및 Calendar 통합 기능이 추가되었습니다! 🚀**

## (선택 사항): Cognito로 Identity Provider 생성

추가 Identity 관리 기능이 필요한 경우 Cognito 기반 Identity Provider도 구성할 수 있습니다. 이 섹션에서는 Amazon Cognito를 사용하여 사용자 지정 OAuth2 Provider를 생성하는 방법을 보여 줍니다.


In [ ]:
cognito_provider_name = "customersupport-gateways-cognito"

try:
    print("📥 Fetching Cognito configuration from SSM...")

    client_id = get_ssm_parameter("/app/customersupport/agentcore/client_id")
    print(f"✅ Retrieved client ID: {client_id}")

    client_secret = get_ssm_parameter("/app/customersupport/agentcore/cognito_secret")
    print(f"✅ Retrieved client secret: {client_secret[:4]}***")

    issuer = get_ssm_parameter("/app/customersupport/agentcore/cognito_discovery_url")
    auth_url = get_ssm_parameter("/app/customersupport/agentcore/cognito_auth_url")
    token_url = get_ssm_parameter("/app/customersupport/agentcore/cognito_token_url")

    print(f"✅ Issuer: {issuer}")
    print(f"✅ Authorization Endpoint: {auth_url}")
    print(f"✅ Token Endpoint: {token_url}")

    print("⚙️  Creating OAuth2 credential provider...")

    cognito_provider = identity_client.create_oauth2_credential_provider(
        name=cognito_provider_name,
        credentialProviderVendor="CustomOauth2",
        oauth2ProviderConfigInput={
            "customOauth2ProviderConfig": {
                "clientId": client_id,
                "clientSecret": client_secret,
                "oauthDiscovery": {
                    "authorizationServerMetadata": {
                        "issuer": issuer,
                        "authorizationEndpoint": auth_url,
                        "tokenEndpoint": token_url,
                        "responseTypes": ["code", "token"],
                    }
                },
            }
        },
    )

    provider_arn = cognito_provider["credentialProviderArn"]
    print(provider_arn)
except Exception as e:
    print(f"❌ Error creating Cognito credential provider: {str(e)}")

In [ ]:
response = identity_client.list_oauth2_credential_providers(maxResults=20)
providers = response.get("credentialProviders", [])
print(providers)

## 정리

In [ ]:
#  try:
#     print(f"🗑️  Deleting Google OAuth2 credential provider: {google_provider_name}")
#     identity_client.delete_oauth2_credential_provider(name=google_provider_name)
#     print("✅ Google OAuth2 credential provider deleted successfully")
# except Exception as e:
#     print(f"❌ Error deleting credential provider: {str(e)}")

In [ ]:
#  try:
#     print(f"🗑️  Deleting Cognito OAuth2 credential provider: {cognito_provider_name}")
#     identity_client.delete_oauth2_credential_provider(name=cognito_provider_name)
#     print("✅ Cognito credential provider deleted successfully")
# except Exception as e:
#     print(f"❌ Error deleting credential provider: {str(e)}")